<a href="https://colab.research.google.com/github/PeroronShine/education_fefu_2/blob/main/cubernetic/%D0%BA%D0%B8%D0%B1%D0%B5%D1%80%D0%BD%D0%B5%D1%82%D0%B8%D0%BA%D0%B04_%D1%85%D1%8D%D0%BC%D0%BC%D0%B8%D0%BD%D0%B3_%D0%B3%D0%BE%D0%BB%D0%B5%D0%B9_%D1%80%D0%B0%D0%B7%D0%B1%D0%B8%D0%B5%D0%BD%D0%B8%D0%B5%20%D0%BD%D0%B0%20%D1%88%D0%B0%D1%80%D1%8B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
def calculate_parity_bits_length(m):
    r = 0
    while (1 << r) < m + r + 1:
        r += 1
    return r

def encode_hamming_block(data_bits, verbose=True):
    m = len(data_bits)
    r = calculate_parity_bits_length(m)
    n = m + r

    if verbose:
        print(f"🔹 Кодирование блока данных: {data_bits} → длина данных = {m}")
        print(f"🔹 Требуется r = {r} контрольных битов → общий размер блока n = {n}")

    # Инициализируем закодированный блок
    encoded = [0] * n
    j = 0
    for i in range(n):
        if (i + 1) & i != 0:  # не степень двойки → информационный бит
            encoded[i] = data_bits[j]
            j += 1

    if verbose:
        print(f"\n🔹 После размещения информационных битов (позиции 3,5,6,7...):")
        print(f"   Позиции (1-based): {[i+1 for i in range(n)]}")
        print(f"   Блок (временно):    {encoded}")
        print()

    # Вычисляем каждый контрольный бит
    for i in range(r):
        parity_pos = (1 << i) - 1
        parity_bit_name = f"P{1 << i}"
        parity = 0
        participating_bits = []  # Список (позиция, значение)

        for j in range(n):
            pos = j + 1
            if pos & (1 << i):  # если i-й бит в номере позиции установлен
                participating_bits.append((pos, encoded[j]))
                parity ^= encoded[j]

        # Присваиваем вычисленное значение
        encoded[parity_pos] = parity

        if verbose:
            pos_list = [f"{pos}({val})" for pos, val in participating_bits]
            print(f"🧮 Вычисление {parity_bit_name} (контрольный бит в позиции {parity_pos + 1}):")
            print(f"   Участвуют биты в позициях: {', '.join(pos_list)}")
            print(f"   XOR = {' ^ '.join(str(val) for _, val in participating_bits)} = {parity}")
            print(f"   → {parity_bit_name} = {parity}")
            print()

    if verbose:
        print(f"✅ Окончательный закодированный блок: {encoded}")
        print("-" * 50)

    return encoded

def decode_hamming_block(received, verbose=True):
    n = len(received)
    r = 0
    while (1 << r) < n + 1:
        r += 1
    m = n - r

    if verbose:
        print(f"🔹 Декодирование блока длины {n}: {received}")
        print(f"🔹 Определяем: контрольных битов r = {r}, информационных m = {m}")
        print(f"   Позиции: {[i+1 for i in range(n)]}")
        print()

    # Вычисляем где ошибка: проверяем каждый контрольный бит
    syndrome = 0
    syndrome_bits_info = []  # для наглядного вывода

    for i in range(r):
        parity_bit_power = 1 << i  # 1, 2, 4, 8...
        parity_bit_name = f"P{parity_bit_power}"
        parity_pos = parity_bit_power - 1  # 0-based позиция контрольного бита

        parity_computed = 0
        participating_bits = []

        for j in range(n):
            pos = j + 1  # 1-based
            if pos & parity_bit_power:  # если бит участвует в проверке Pi
                participating_bits.append((pos, received[j]))
                parity_computed ^= received[j]

        # Если чётность не нулевая → ошибка в этой группе
        if parity_computed != 0:
            syndrome += parity_bit_power
            syndrome_bits_info.append((parity_bit_name, parity_computed))

        if verbose:
            pos_list = [f"{pos}({val})" for pos, val in participating_bits]
            print(f"🧮 Проверка {parity_bit_name} (должно быть чётное число единиц):")
            print(f"   Биты в позициях: {', '.join(pos_list)}")
            print(f"   XOR = {' ^ '.join(str(val) for _, val in participating_bits)} = {parity_computed}")
            if parity_computed == 0:
                print(f"   → Чётность соблюдена ✅")
            else:
                print(f"   → Нарушена чётность ❌")
            print()

    if verbose:
        if syndrome == 0:
            print("✅ 0 → ошибок не обнаружено.")
        else:
            print(f"❗ Ошибка в символе №{syndrome} ")

    # Исправление ошибки (если позиция в пределах блока)
    error_pos_1based = syndrome
    error_pos_0based = syndrome - 1
    if syndrome != 0:
        if 1 <= error_pos_1based <= n:
            if verbose:
                old_val = received[error_pos_0based]
                print(f"Исправляем бит в позиции {error_pos_1based}: {old_val} → {1 - old_val}")
            received[error_pos_0based] ^= 1
            if verbose:
                print(f"   Блок после исправления: {received}")
        else:
            if verbose:
                print(f"Ошибка указывает на позицию {error_pos_1based}, но блок имеет длину {n} → ошибка неисправима.")
    elif verbose:
        print("   Блок остаётся без изменений.")

    # Извлекаем информационные биты (все, кроме позиций 1,2,4,8,...)
    data = []
    info_positions = []
    for i in range(n):
        pos = i + 1
        if pos & (pos - 1) != 0:  # не степень двойки → информационный бит
            data.append(received[i])
            info_positions.append(pos)

    if verbose:
        print(f"\nИзвлекаем информационные биты из позиций: {info_positions}")
        print(f"   Результат: {data[:m]}")
        print("-" * 50)

    return data[:m]

def hamming_encode(message_bits, verbose=True):
    original_length = len(message_bits)
    # размер блока на которые делим исходное сообщение
    chunk_size = 4
    padded = message_bits + [0] * ((-len(message_bits)) % chunk_size)

    if verbose:
        print(f"=== КОДИРОВАНИЕ ===")
        print(f"Исходное сообщение: {message_bits} (длина = {original_length})")
        if len(padded) != original_length:
            print(f"Дополнено нулями до длины {len(padded)}: {padded}")
        else:
            print("Сообщение уже кратно 4 — дополнение не требуется.")

    encoded = []
    for idx, chunk in enumerate([padded[i:i+4] for i in range(0, len(padded), 4)]):
        if verbose:
            print(f"\n--- Блок {idx + 1} ---")
        encoded_block = encode_hamming_block(chunk, verbose=verbose)
        encoded.extend(encoded_block)

    if verbose:
        print(f"\nПолное сообщение:    {encoded}\n")
        print("-" * 50)

    return encoded, original_length

def hamming_decode(encoded_bits, original_length, verbose=True):
    block_size = 7
    decoded_bits = []

    if verbose:
        print(f"Ожидаемая длина исходного сообщения: {original_length}")

    blocks = [encoded_bits[i:i+block_size] for i in range(0, len(encoded_bits), block_size)]
    for idx, block in enumerate(blocks):
        block = (block + [0] * block_size)[:block_size]
        if verbose:
            print(f"\n--- Блок {idx + 1} ---")
        data = decode_hamming_block(block, verbose=verbose)
        decoded_bits.extend(data)

    result = decoded_bits[:original_length]
    if verbose:
        print(f"✅ Восстановленное сообщение: {result}\n")

    return result

if __name__ == "__main__":
    message = [1, 0, 1, 1]

    print("🔤 Входное сообщение (биты):", message, "\n")

    encoded, orig_len = hamming_encode(message, verbose=True)
    corrupted = encoded[:]

    # индекс символа для ошибки
    error_index = 4
    corrupted[error_index] ^= 1

    print("💥 Вносим ошибку в позицию", error_index + 1)
    print("Сообщение с ошибкой:", corrupted, "\n")

    decoded = hamming_decode(corrupted, orig_len, verbose=True)

    print("Исходное: ", message)
    print("Восстановленное:", decoded)

🔤 Входное сообщение (биты): [1, 0, 1, 1] 

=== КОДИРОВАНИЕ ===
Исходное сообщение: [1, 0, 1, 1] (длина = 4)
Сообщение уже кратно 4 — дополнение не требуется.

--- Блок 1 ---
🔹 Кодирование блока данных: [1, 0, 1, 1] → длина данных = 4
🔹 Требуется r = 3 контрольных битов → общий размер блока n = 7

🔹 После размещения информационных битов (позиции 3,5,6,7...):
   Позиции (1-based): [1, 2, 3, 4, 5, 6, 7]
   Блок (временно):    [0, 0, 1, 0, 0, 1, 1]

🧮 Вычисление P1 (контрольный бит в позиции 1):
   Участвуют биты в позициях: 1(0), 3(1), 5(0), 7(1)
   XOR = 0 ^ 1 ^ 0 ^ 1 = 0
   → P1 = 0

🧮 Вычисление P2 (контрольный бит в позиции 2):
   Участвуют биты в позициях: 2(0), 3(1), 6(1), 7(1)
   XOR = 0 ^ 1 ^ 1 ^ 1 = 1
   → P2 = 1

🧮 Вычисление P4 (контрольный бит в позиции 4):
   Участвуют биты в позициях: 4(0), 5(0), 6(1), 7(1)
   XOR = 0 ^ 0 ^ 1 ^ 1 = 0
   → P4 = 0

✅ Окончательный закодированный блок: [0, 1, 1, 0, 0, 1, 1]
--------------------------------------------------

Полное сообщение:  

In [ ]:
def encode_hamming_extended_8_4(data_bits, verbose=True):
    if len(data_bits) != 4:
        raise ValueError("Для (8,4)-кода требуется ровно 4 информационных бита.")

    d1, d2, d3, d4 = data_bits

    if verbose:
        print(f"🔹 Кодирование блока данных: {data_bits}")
        print("   Информационные биты: D1={}, D2={}, D3={}, D4={}".format(d1, d2, d3, d4))
        print("   Позиции (1-based): 1=P1, 2=P2, 3=D1, 4=P3, 5=D2, 6=D3, 7=D4, 8=P0 (общая чётность)")

    # Вычисляем контрольные биты для (7,4)-кода
    p1 = d1 ^ d2 ^ d4
    p2 = d1 ^ d3 ^ d4
    p3 = d2 ^ d3 ^ d4

    block_7 = [p1, p2, d1, p3, d2, d3, d4]

    if verbose:
        print("\n   После размещения информационных битов (позиции 3,5,6,7):")
        print("   Позиции (1-based): [1, 2, 3, 4, 5, 6, 7]")
        print("   Блок (7 бит):      {}".format(block_7))
        print()

        # Вывод вычисления P1
        bits_p1 = [(1, p1), (3, d1), (5, d2), (7, d4)]
        vals_p1 = [str(b) for _, b in bits_p1]
        print("   Вычисление P1 (позиция 1): XOR битов в позициях 1,3,5,7")
        print("     Участвующие биты: " + ", ".join(f"{pos}({val})" for pos, val in bits_p1))
        print("     XOR = {} = {}".format(" ^ ".join(vals_p1), p1))
        print("     → P1 = {}".format(p1))

        # P2
        bits_p2 = [(2, p2), (3, d1), (6, d3), (7, d4)]
        vals_p2 = [str(b) for _, b in bits_p2]
        print("   Вычисление P2 (позиция 2): XOR битов в позициях 2,3,6,7")
        print("     Участвующие биты: " + ", ".join(f"{pos}({val})" for pos, val in bits_p2))
        print("     XOR = {} = {}".format(" ^ ".join(vals_p2), p2))
        print("     → P2 = {}".format(p2))

        # P3
        bits_p3 = [(4, p3), (5, d2), (6, d3), (7, d4)]
        vals_p3 = [str(b) for _, b in bits_p3]
        print("   Вычисление P3 (позиция 4): XOR битов в позициях 4,5,6,7")
        print("     Участвующие биты: " + ", ".join(f"{pos}({val})" for pos, val in bits_p3))
        print("     XOR = {} = {}".format(" ^ ".join(vals_p3), p3))
        print("     → P3 = {}".format(p3))

    # Добавляем общий бит чётности P0 (позиция 8)
    total_xor = p1 ^ p2 ^ d1 ^ p3 ^ d2 ^ d3 ^ d4
    p0 = total_xor  # чтобы XOR всех 8 битов был 0 (чётная общая чётность)

    encoded = block_7 + [p0]

    if verbose:
        all_bits = [(i+1, encoded[i]) for i in range(8)]
        vals_all = [str(b) for _, b in all_bits]
        print("   Вычисление P0 (позиция 8, общая чётность): XOR всех 8 битов должен быть 0")
        print("     Текущий XOR первых 7 битов = {}".format(total_xor))
        print("     → P0 = {}".format(p0))
        print("     Полный блок (8 бит): {}".format(encoded))
        print("-" * 50)

    return encoded


def decode_hamming_extended_8_4(received, verbose=True):
    if len(received) < 8:
        received = (received + [0] * 8)[:8]
    else:
        received = received[:8]

    p1, p2, d1, p3, d2, d3, d4, p0 = received

    if verbose:
        print("🔹 Декодирование блока длины 8: {}".format(received))
        print("   Позиции: [1, 2, 3, 4, 5, 6, 7, 8]")

    # Проверочные уравнения (как при кодировании)
    s1 = p1 ^ d1 ^ d2 ^ d4
    s2 = p2 ^ d1 ^ d3 ^ d4
    s3 = p3 ^ d2 ^ d3 ^ d4

    syndrome = s1 + (s2 << 1) + (s3 << 2)  # от 0 до 7

    overall_parity = p1 ^ p2 ^ d1 ^ p3 ^ d2 ^ d3 ^ d4 ^ p0

    if verbose:
        print("   Проверка P1 (позиции 1,3,5,7): XOR = {} → s1 = {}".format(p1 ^ d1 ^ d2 ^ d4, s1))
        print("   Проверка P2 (позиции 2,3,6,7): XOR = {} → s2 = {}".format(p2 ^ d1 ^ d3 ^ d4, s2))
        print("   Проверка P3 (позиции 4,5,6,7): XOR = {} → s3 = {}".format(p3 ^ d2 ^ d3 ^ d4, s3))
        print("   Синдром (S3 S2 S1) = {}{}{} → {}".format(s3, s2, s1, syndrome))
        print("   Общая чётность (XOR всех 8 битов) = {}".format(overall_parity))

    corrected = received[:]

    if syndrome == 0:
        if overall_parity == 0:
            if verbose:
                print("   Ошибок не обнаружено.")
        else:
            if verbose:
                print("   Ошибка только в бите общей чётности (позиция 8). Исправляем.")
            corrected[7] ^= 1
    else:
        if overall_parity == 1:
            error_pos = syndrome  # 1-based
            if 1 <= error_pos <= 8:
                if verbose:
                    old_val = corrected[error_pos - 1]
                    print("   Обнаружена одиночная ошибка в позиции {}. Исправляем: {} → {}".format(
                        error_pos, old_val, 1 - old_val))
                corrected[error_pos - 1] ^= 1
            else:
                if verbose:
                    print("   Синдром указывает на позицию {}, выходящую за пределы блока.".format(error_pos))
        else:
            if verbose:
                print("   Обнаружено ДВЕ ошибки. Исправление невозможно.")

    # Извлекаем информационные биты: позиции 3,5,6,7 → индексы 2,4,5,6
    data = [corrected[i] for i in [2, 4, 5, 6]]

    if verbose:
        print("   Извлечённые информационные биты (позиции 3,5,6,7): {}".format(data))
        print("-" * 50)

    return data


def hamming_encode(message_bits, verbose=True):
    original_length = len(message_bits)
    chunk_size = 4
    padded = message_bits + [0] * ((-len(message_bits)) % chunk_size)

    if verbose:
        print("=== КОДИРОВАНИЕ (8,4) ===")
        print("Исходное сообщение: {} (длина = {})".format(message_bits, original_length))
        if len(padded) != original_length:
            print("Дополнено нулями до длины {}: {}".format(len(padded), padded))
        else:
            print("Сообщение уже кратно 4 — дополнение не требуется.")

    encoded = []
    blocks = [padded[i:i+4] for i in range(0, len(padded), 4)]
    for idx, chunk in enumerate(blocks):
        if verbose:
            print("\n--- Блок {} ---".format(idx + 1))
        block_encoded = encode_hamming_extended_8_4(chunk, verbose=verbose)
        encoded.extend(block_encoded)

    if verbose:
        print("\nЗакодированное сообщение: {}".format(encoded))
        print("-" * 50)

    return encoded, original_length


def hamming_decode(encoded_bits, original_length, verbose=True):
    block_size = 8
    decoded_bits = []

    if verbose:
        print("Ожидаемая исходная длина: {}".format(original_length))

    blocks = [encoded_bits[i:i+block_size] for i in range(0, len(encoded_bits), block_size)]
    for idx, block in enumerate(blocks):
        if verbose:
            print("\n--- Блок {} ---".format(idx + 1))
        data = decode_hamming_extended_8_4(block, verbose=verbose)
        decoded_bits.extend(data)

    result = decoded_bits[:original_length]
    if verbose:
        print("\nВосстановленное сообщение: {}".format(result))

    return result


if __name__ == "__main__":
    message = [1, 0, 0, 1]

    print("Входное сообщение (биты):", message, "\n")

    encoded, orig_len = hamming_encode(message, verbose=True)

    # Одиночная ошибка
    corrupted = encoded[:]
    error_index = 0
    corrupted[error_index] ^= 1
    print("Сообщение с ошибкой:     ", corrupted)

    decoded = hamming_decode(corrupted, orig_len, verbose=True)
    print("\nИсходное:     ", message)
    print("Восстановлено:", decoded)

    print("\n" + "="*60 + "\n")

    # Две ошибки
    corrupted2 = encoded[:]
    corrupted2[0] ^= 1
    corrupted2[2] ^= 1
    print("Сообщение с ошибками:", corrupted2)

    decoded2 = hamming_decode(corrupted2, orig_len, verbose=True)

Входное сообщение (биты): [1, 0, 0, 1] 

=== КОДИРОВАНИЕ (8,4) ===
Исходное сообщение: [1, 0, 0, 1] (длина = 4)
Сообщение уже кратно 4 — дополнение не требуется.

--- Блок 1 ---
🔹 Кодирование блока данных: [1, 0, 0, 1]
   Информационные биты: D1=1, D2=0, D3=0, D4=1
   Позиции (1-based): 1=P1, 2=P2, 3=D1, 4=P3, 5=D2, 6=D3, 7=D4, 8=P0 (общая чётность)

   После размещения информационных битов (позиции 3,5,6,7):
   Позиции (1-based): [1, 2, 3, 4, 5, 6, 7]
   Блок (7 бит):      [0, 0, 1, 1, 0, 0, 1]

   Вычисление P1 (позиция 1): XOR битов в позициях 1,3,5,7
     Участвующие биты: 1(0), 3(1), 5(0), 7(1)
     XOR = 0 ^ 1 ^ 0 ^ 1 = 0
     → P1 = 0
   Вычисление P2 (позиция 2): XOR битов в позициях 2,3,6,7
     Участвующие биты: 2(0), 3(1), 6(0), 7(1)
     XOR = 0 ^ 1 ^ 0 ^ 1 = 0
     → P2 = 0
   Вычисление P3 (позиция 4): XOR битов в позициях 4,5,6,7
     Участвующие биты: 4(1), 5(0), 6(0), 7(1)
     XOR = 1 ^ 0 ^ 0 ^ 1 = 1
     → P3 = 1
   Вычисление P0 (позиция 8, общая чётность): XOR всех

In [ ]:
import numpy as np

def golay_matrices():
    """
    Возвращает порождающую G (12x24) и проверочную H (12x24) матрицы
    расширенного двоичного кода Голея (24,12,8) в систематической форме.
    G = [I | P], H = [P^T | I]
    """
    # P — матрица 12x12, взята из стандартной формы кода Голея
    P = np.array([
        [1,1,0,1,1,1,0,0,0,1,0,1],
        [1,0,1,1,1,0,0,0,1,0,1,1],
        [0,1,1,1,0,0,0,1,0,1,1,1],
        [1,1,1,0,0,0,1,0,1,1,0,1],
        [1,1,0,0,0,1,0,1,1,0,1,1],
        [1,0,0,0,1,0,1,1,0,1,1,1],
        [0,0,0,1,0,1,1,0,1,1,1,1],
        [0,0,1,0,1,1,0,1,1,1,1,0],
        [0,1,0,1,1,0,1,1,1,1,0,0],
        [1,0,1,1,0,1,1,1,1,0,0,0],
        [0,1,1,0,1,1,1,1,0,0,0,1],
        [1,1,0,1,1,1,1,0,0,0,1,0]
    ], dtype=int) % 2

    I12 = np.eye(12, dtype=int)
    G = np.concatenate([I12, P], axis=1) % 2
    H = np.concatenate([P.T, I12], axis=1) % 2

    return G, H

def encode_golay(u, G):
    """Матричное кодирование: c = u @ G mod 2"""
    u = np.array(u, dtype=int)
    c = (u @ G) % 2
    return c

def syndrome(r, H):
    """Вычислить синдром: s = r @ H^T mod 2"""
    r = np.array(r, dtype=int)
    s = (r @ H.T) % 2
    return s

def correct_golay(r, H):
    """
    Декодирование расширенного кода Голея (24,12,8)
    Возвращает: восстановленное слово, вектор ошибок, флаг успеха
    """
    n = 24
    t = 3  # код исправляет до 3 ошибок
    r = np.array(r, dtype=int)
    s = syndrome(r, H)

    print(f"\nСиндром: {s.tolist()}")

    # Если синдром нулевой — ошибок нет
    if not s.any():
        print("Синдром нулевой → ошибок не обнаружено.")
        return r, np.zeros(n, dtype=int), True

    # Попробуем найти ошибку весом ≤3
    # Перебираем все возможные векторы ошибок веса 1, 2, 3
    from itertools import combinations

    # Вес 1
    for i in range(n):
        e = np.zeros(n, dtype=int)
        e[i] = 1
        if np.array_equal(syndrome(e, H), s):
            print(f"✅ Найдена ОДИНОЧНАЯ ошибка в позиции {i}.")
            c_correct = (r + e) % 2
            return c_correct, e, True

    # Вес 2
    for i, j in combinations(range(n), 2):
        e = np.zeros(n, dtype=int)
        e[i] = e[j] = 1
        if np.array_equal(syndrome(e, H), s):
            print(f"✅ Найдена ДВОЙНАЯ ошибка в позициях {i}, {j}.")
            c_correct = (r + e) % 2
            return c_correct, e, True

    # Вес 3
    for combo in combinations(range(n), 3):
        e = np.zeros(n, dtype=int)
        e[list(combo)] = 1
        if np.array_equal(syndrome(e, H), s):
            print(f"✅ Найдена ТРОЙНАЯ ошибка в позициях {combo}.")
            c_correct = (r + e) % 2
            return c_correct, e, True

    # Ошибок >3 — неисправимо
    print("Обнаружена ошибка кратности ≥4 — исправление невозможно.")
    return r, None, False

def main():
    print("=== КОД ГОЛЕЯ (24,12,8) ===")

    # Ввод информационного слова
    print("\nВведите информационное слово длины 12 (только 0 и 1):")
    u_str = input().strip()
    if len(u_str) != 12 or not set(u_str).issubset({'0','1'}):
        raise ValueError("Длина должна быть 12, символы — только 0 и 1.")
    u = [int(ch) for ch in u_str]
    print(f"Исходное u = {u}")

    # Генерация матриц
    G, H = golay_matrices()
    print("\nПорождающая матрица G (12×24):")
    print(G)
    print("\nПроверочная матрица H (12×24):")
    print(H)

    # Кодирование
    c = encode_golay(u, G)
    print(f"\nКодовое слово c = {c.tolist()}")

    # Внесение ошибок
    print("\nХотите внести ошибки? (y/n):")
    if input().strip().lower() == 'y':
        num = int(input("Сколько ошибок внести (1–4)? "))
        positions = []
        for i in range(num):
            pos = int(input(f"Позиция ошибки {i+1} (0–23): "))
            if not (0 <= pos < 24):
                raise ValueError("Позиция вне диапазона")
            if pos in positions:
                raise ValueError("Позиция уже указана")
            positions.append(pos)
        r = c.copy()
        for p in positions:
            r[p] ^= 1
        print(f"\nОшибки внесены в позиции: {positions}")
        print(f"Искажённое слово r = {r.tolist()}")
    else:
        r = c.copy()
        print("\nОшибки не внесены.")

    # Декодирование
    c_rec, e_vec, success = correct_golay(r, H)

    if success:
        u_rec = c_rec[:12]  # систематический вид
        print(f"\nВосстановленное информационное слово: {u_rec.tolist()}")
        if np.array_equal(u, u_rec):
            print("✅ Декодирование успешно!")
        else:
            print("❌ Ошибка декодирования!")
    else:
        print("\nДекодирование не выполнено (ошибка неисправима).")

    # Анализ
    weight = np.sum(e_vec) if e_vec is not None else len(positions) if 'positions' in locals() else 0
    print(f"\nАнализ:")
    if weight <= 3:
        print(f"  • Ошибок: {weight} → исправимо (d=8, t=⌊(d−1)/2⌋=3)")
    else:
        print(f"  • Ошибок: {weight} → неисправимо, но обнаружимо (если ≤4)")
        if weight == 4:
            print("    (расширенный код Голея обнаруживает все 4-кратные ошибки)")

if __name__ == "__main__":
    main()

=== КОД ГОЛЕЯ (24,12,8) ===

Введите информационное слово длины 12 (только 0 и 1):
111100001111
Исходное u = [1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1]

Порождающая матрица G (12×24):
[[1 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1 1 1 0 0 0 1 0 1]
 [0 1 0 0 0 0 0 0 0 0 0 0 1 0 1 1 1 0 0 0 1 0 1 1]
 [0 0 1 0 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 1 0 1 1 1]
 [0 0 0 1 0 0 0 0 0 0 0 0 1 1 1 0 0 0 1 0 1 1 0 1]
 [0 0 0 0 1 0 0 0 0 0 0 0 1 1 0 0 0 1 0 1 1 0 1 1]
 [0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 1 0 1 1 0 1 1 1]
 [0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 1 0 1 1 0 1 1 1 1]
 [0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 1 1 0 1 1 1 1 0]
 [0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 1 1 0 1 1 1 1 0 0]
 [0 0 0 0 0 0 0 0 0 1 0 0 1 0 1 1 0 1 1 1 1 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 0 0 1 1 0 1 1 1 1 0 0 0 1]
 [0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 1 1 1 1 0 0 0 1 0]]

Проверочная матрица H (12×24):
[[1 1 0 1 1 1 0 0 0 1 0 1 1 0 0 0 0 0 0 0 0 0 0 0]
 [1 0 1 1 1 0 0 0 1 0 1 1 0 1 0 0 0 0 0 0 0 0 0 0]
 [0 1 1 1 0 0 0 1 0 1 1 0 0 0 1 0 0 0 0 0 0 0 0 0]
 [1 1 1 0 0 0 1 0 1 1 0 

КОД ХЭММИНГА


In [ ]:
import numpy as np

def find_min_m_for_k(k_needed):
    m = 1
    while True:
        k = (1 << m) - m - 1
        if k >= k_needed:
            return m, k, (1 << m) - 1
        m += 1
        if m > 10:
            raise ValueError("Требуемая длина слишком велика для m ≤ 10")

def generate_hamming_matrices(m):
    n = (1 << m) - 1
    k = n - m

    cols = []
    for i in range(1, n + 1):
        col = [(i >> j) & 1 for j in reversed(range(m))]
        cols.append(col)
    H_full = np.array(cols).T % 2

    ident_positions = []
    I_m = np.eye(m, dtype=int)
    for i in range(m):
        target = I_m[:, i]
        found = False
        for j in range(n):
            if np.array_equal(H_full[:, j], target):
                ident_positions.append(j)
                found = True
                break
        if not found:
            raise RuntimeError(f"Не найден столбец для единичного вектора {i}")

    all_indices = set(range(n))
    parity_set = set(ident_positions)
    info_indices = sorted(all_indices - parity_set)

    new_order = info_indices + ident_positions
    H_sys = H_full[:, new_order]

    P = H_sys[:, :k]
    G_sys = np.concatenate([np.eye(k, dtype=int), P.T], axis=1) % 2

    return H_sys, G_sys, n, k, new_order

def encode_hamming(u, G):
    u = np.array(u, dtype=int)
    return (u @ G) % 2

def decode_hamming_with_steps(r, H, n, k, error_positions_true=None):
    r = np.array(r, dtype=int)
    print("\n=== ДЕКОДИРОВАНИЕ ===")
    print(f"Принятое слово r = {r.tolist()}")

    s = (r @ H.T) % 2
    print(f"1. Синдром s = r · H^T = {s.tolist()}")

    if not s.any():
        print("2. Синдром нулевой → ошибок не обнаружено.")
        c_correct = r
        corrected_positions = []
    else:
        print("2. Синдром ненулевой → ищем позицию ошибки...")
        error_pos = None
        for j in range(n):
            if np.array_equal(H[:, j], s):
                error_pos = j
                print(f"   Ошибка найдена в позиции {j}.")
                break

        if error_pos is not None:
            c_correct = r.copy()
            c_correct[error_pos] ^= 1
            print(f"3. Исправленное кодовое слово: {c_correct.tolist()}")
            corrected_positions = [error_pos]
        else:
            print("3. Синдром не совпадает ни с одним столбцом H → обнаружено более одной ошибки.")
            print("   Исправление невозможно — ошибка НЕИСПРАВИМА.")
            c_correct = r
            corrected_positions = []

    u_hat = c_correct[:k]
    print(f"4. Восстановленное информационное слово u = {u_hat.tolist()}")

    # Анализ корректности (если известны истинные позиции ошибок)
    if error_positions_true is not None:
        print("\n=== АНАЛИЗ ИСПРАВЛЕНИЯ ===")
        detected_errors = set(corrected_positions)
        true_errors = set(error_positions_true)
        if len(true_errors) == 1 and detected_errors == true_errors:
            print("✅ Одиночная ошибка успешно исправлена.")
        elif len(true_errors) >= 2:
            print(f"⚠️  Обнаружено {len(true_errors)} ошибок — исправление невозможно.")
            if detected_errors:
                print(f"   Декодер ошибочно попытался исправить позицию {list(detected_errors)[0]}.")
        else:
            print("❌ Ошибка декодирования: позиция определена неверно.")

    return u_hat

def main():
    print("Введите информационное слово (только 0 и 1):")
    u_str = input().strip()
    if not u_str or not set(u_str).issubset({'0', '1'}):
        raise ValueError("Некорректная строка. Используйте только 0 и 1.")

    k_input = len(u_str)
    u_input = [int(ch) for ch in u_str]
    print(f"\nИнформационное слово длины {k_input}: {u_input}")

    m, k, n = find_min_m_for_k(k_input)
    print(f"\nПодобран параметр m = {m} → код ({n}, {k}) с d = 3")

    if k_input < k:
        u_padded = u_input + [0] * (k - k_input)
        print(f"\nСлово дополнено нулями до длины k = {k}: {u_padded}")
    else:
        u_padded = u_input

    H, G, n_real, k_real, order = generate_hamming_matrices(m)
    assert n_real == n and k_real == k

    print(f"\nПорождающая матрица G ({k}×{n}):")
    print(G)
    print(f"\nПроверочная матрица H ({m}×{n}):")
    print(H)

    c = encode_hamming(u_padded, G)
    print(f"\n=== КОДИРОВАНИЕ ===")
    print(f"Кодовое слово c = u · G = {c.tolist()}")

    print("\nХотите внести ошибки? (1/0):")
    if input().strip().lower() == '1':
        num_errors = int(input(f"Сколько ошибок внести (1–{n})? "))
        if num_errors < 1 or num_errors > n:
            raise ValueError("Недопустимое число ошибок")
        error_positions = []
        for i in range(num_errors):
            pos = int(input(f"Позиция ошибки {i+1} (0–{n-1}): "))
            if not (0 <= pos < n):
                raise ValueError("Позиция вне диапазона")
            if pos in error_positions:
                raise ValueError("Позиция уже указана")
            error_positions.append(pos)

        r = c.copy()
        for pos in error_positions:
            r[pos] ^= 1
        print(f"\nВнесены ошибки в позиции: {error_positions}")
        print(f"Искажённое слово: {r.tolist()}")
    else:
        r = c.copy()
        error_positions = []
        print("\nОшибки не внесены.")

    try:
        u_rec = decode_hamming_with_steps(r, H, n, k, error_positions_true=error_positions)
        if np.array_equal(u_rec, u_padded):
            print("\n✅ Восстановление прошло успешно: u_rec == u_padded")
        else:
            print("\n❌ Восстановление не удалось.")
    except ValueError as e:
        print(f"\n❌ Ошибка: {e}")

if __name__ == "__main__":
    main()

Введите информационное слово (только 0 и 1):
1010101011111101

Информационное слово длины 16: [1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1]

Подобран параметр m = 5 → код (31, 26) с d = 3

Слово дополнено нулями до длины k = 26: [1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Порождающая матрица G (26×31):
[[1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1]
 [0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1]
 [0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0]
 [0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1]
 [0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1]
 [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0]
 [0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 1]
 [0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0]
 [0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1]
 [0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0]
 [0 0 0

разбиение пространства на сферы

In [ ]:
from itertools import combinations

def hamming_distance(v1, v2):
    """Вычисляет расстояние Хемминга между двумя бинарными векторами."""
    return sum(a != b for a, b in zip(v1, v2))

def hamming_ball(center, radius, n):
    """Возвращает множество всех векторов в F_2^n, находящихся на расстоянии <= radius от center."""
    ball = []
    for i in range(n + 1):
        if i <= radius:
            # Все векторы с весом i, отличающиеся от center в i позициях
            for positions in combinations(range(n), i):
                vec = list(center)
                for pos in positions:
                    vec[pos] = 1 - vec[pos]  # инвертируем бит
                ball.append(tuple(vec))
    return set(ball)

def volume_of_hamming_ball(n, t):
    """Объём шара Хемминга радиуса t в F_2^n."""
    vol = 0
    for i in range(t + 1):
        # Число способов выбрать i позиций из n
        c = 1
        for j in range(i):
            c = c * (n - j) // (j + 1)
        vol += c
    return vol

def is_perfect_code(n, t):
    """
    Проверяет, существует ли совершенный код (разбиение пространства на непересекающиеся шары радиуса t).
    Возвращает:
      - True/False
      - Список центров (если удалось построить или известен)
    """
    total_vectors = 2 ** n
    ball_size = volume_of_hamming_ball(n, t)

    if total_vectors % ball_size != 0:
        return False, []

    # 1. Тривиальный случай: t == 0 -> каждый вектор — центр
    if t == 0:
        centers = [tuple([0]*n)]
        return True, centers

    # 2. Тривиальный случай: t >= n -> вся сфера = всё пространство
    if t >= n:
        centers = [tuple([0]*n)]
        return True, centers

    # 3. Повторные коды нечётной длины: t = (n-1)/2, n нечётно, |C| = 2
    if n % 2 == 1 and t == (n - 1) // 2:
        centers = [tuple([0]*n), tuple([1]*n)]
        return True, centers

    # 4. Коды Хэмминга: t=1, n = 2^r - 1, r>=2
    if t == 1:
        r = 0
        while (1 << r) - 1 < n:
            r += 1
        if (1 << r) - 1 == n and r >= 2:
            return True, [tuple([0]*n)]

    # 5. Бинарный код Голея: t=3, n=23
    if t == 3 and n == 23:
        return True, [tuple([0]*23)]
    return False, []

def main():
    print("=== Проверка возможности разбиения F_2^n на шары Хемминга радиуса t ===")
    n = 2
    t = 7

    exists, centers = is_perfect_code(n, t)
    vol = volume_of_hamming_ball(n, t)
    total = 2 ** n

    if not exists:
        print(f"\nРазбиение невозможно для n={n}, t={t}.")
        return

    expected_centers_count = total // vol

    centers_complete = (len(centers) == expected_centers_count)

    if centers_complete:
        print(f"\nРазбиение возможно. Всего центров: {len(centers)}")
        print("Координаты всех центров:")
        for i, c in enumerate(centers, 1):
            print(f"{i}: {''.join(map(str, c))}")

        print("\n" + "="*50)

        for idx, center in enumerate(centers, 1):
            print(f"\nЦентр {idx}: {''.join(map(str, center))}")
            ball = hamming_ball(center, t, n)
            print("Векторы в шаре:")
            for vec in sorted(ball):
                print(''.join(map(str, vec)))
    else:
        print(f"\nРазбиение существует, но предоставлен неполный список центров.")
        print(f"Ожидается {expected_centers_count} центров, получено {len(centers)}.")
        if centers:
            print("\nИзвестные центры:")
            for i, c in enumerate(centers, 1):
                print(f"{i}: {''.join(map(str, c))}")
            print("\nПример шара вокруг первого центра:")
            center = centers[0]
            ball = hamming_ball(center, t, n)
            print(f"Центр: {''.join(map(str, center))}")
            print("Векторы в шаре (первые 20):")
            for vec in sorted(ball)[:20]:
                print(''.join(map(str, vec)))
            if len(ball) > 20:
                print(f"... и ещё {len(ball) - 20} векторов.")

if __name__ == "__main__":
    main()

=== Проверка возможности разбиения F_2^n на шары Хемминга радиуса t ===

Разбиение возможно. Всего центров: 1
Координаты всех центров:
1: 00


Центр 1: 00
Векторы в шаре:
00
01
10
11
